In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from pathlib import Path

base = Path.cwd().parent
processed = base / "data" / "processed"

WINDOW = 120
MIN_OBS = 36
STALE_THRESHOLD = 0.5
START_YEAR = 2013
END_YEAR = 2024
RIDGE = 1e-8

returns = pd.read_csv(
    processed / "returns.csv",
    index_col=0,
    parse_dates=True
).sort_index()


def solve_min_variance(cov_matrix):
    n = cov_matrix.shape[0]
    Sigma = cov_matrix.values

    res = minimize(
        lambda w: w.T @ Sigma @ w,
        x0=np.ones(n) / n,
        method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1},
        options={"maxiter": 1000, "ftol": 1e-12}
    )

    if not res.success:
        raise ValueError(f"Optimization failed: {res.message}")

    w = pd.Series(res.x, index=cov_matrix.index)
    w[w.abs() < 1e-12] = 0.0
    return w / w.sum()


all_weights = []
all_portfolio_returns = []

for year in range(START_YEAR, END_YEAR + 1):

    dec_date = returns.index[(returns.index.year == year) & (returns.index.month == 12)]
    if len(dec_date) == 0:
        continue
    dec_date = dec_date[-1]

    window_data = returns.loc[:dec_date].tail(WINDOW)
    if len(window_data) < WINDOW:
        continue

    window_data = window_data.loc[:, window_data.count() >= MIN_OBS]
    window_data = window_data.loc[:, window_data.eq(0).sum() / window_data.count() <= STALE_THRESHOLD]
    window_data = window_data.loc[:, window_data.std(skipna=True) > 1e-8]

    if window_data.shape[1] < 2:
        continue

    valid = window_data.cov().dropna(how="any", axis=0).dropna(how="any", axis=1).index
    cov_Y = window_data[valid].cov()
    cov_Y = cov_Y + RIDGE * np.eye(len(cov_Y))

    if cov_Y.shape[0] < 2:
        continue

    try:
        w_rebalanced = solve_min_variance(cov_Y)
    except Exception as e:
        print(f"{year}: optimization failed — {e}")
        continue

    all_weights.append(pd.DataFrame({
        "RebalanceDate": dec_date,
        "Year": year,
        "ISIN": w_rebalanced.index,
        "Weight": w_rebalanced.values
    }))

    print(f"{year}: {len(w_rebalanced)} assets | max weight = {w_rebalanced.max():.4f}")

    next_returns = returns.loc[returns.index.year == year + 1, w_rebalanced.index]
    if next_returns.empty:
        continue

    current_weights = w_rebalanced.copy()

    for dt, r_t in next_returns.iterrows():
        r_t = r_t.reindex(current_weights.index).fillna(0.0)

        if r_t.isna().all():
            continue

        rp_t = (current_weights * r_t).sum()

        all_portfolio_returns.append({
            "Date": dt,
            "RebalanceYear": year,
            "Return": rp_t
        })

        if 1 + rp_t <= 0:
            print(f"Portfolio collapsed at {dt}")
            break

        current_weights = current_weights * (1 + r_t) / (1 + rp_t)
        current_weights[current_weights.abs() < 1e-12] = 0.0
        current_weights = current_weights / current_weights.sum()


weights_df = pd.concat(all_weights, ignore_index=True)

returns_df = pd.DataFrame(all_portfolio_returns)
returns_df["Date"] = pd.to_datetime(returns_df["Date"])
returns_df = returns_df.sort_values("Date")
returns_df["Cumulative"] = (1 + returns_df["Return"]).cumprod() - 1

weights_df.to_csv('output' / "mv_weights.csv", index=False)
returns_df.to_csv('output' / "mv_portfolio_returns.csv", index=False)


2013: 493 assets | max weight = 0.2862
2014: 494 assets | max weight = 0.2254
2015: 495 assets | max weight = 0.2013
2016: 495 assets | max weight = 0.2477
2017: 496 assets | max weight = 0.2398
2018: 496 assets | max weight = 0.4040
2019: 495 assets | max weight = 0.3449
2020: 497 assets | max weight = 0.1803
2021: 498 assets | max weight = 0.2379
2022: 498 assets | max weight = 0.1772
2023: 498 assets | max weight = 0.1379
2024: 498 assets | max weight = 0.0891


NameError: name 'output' is not defined

In [11]:
processed = base / "data" / "processed"

rf_df = pd.read_csv(processed / "risk_free_rate.csv")
rf_df["Date"] = pd.to_datetime(rf_df["Date"].astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
rf_df["RF"] = rf_df["RF"] / 100

perf_df = returns_df.merge(rf_df, on="Date", how="left")
rp = perf_df["Return"]
rf = perf_df["RF"]
excess = rp - rf

metrics = {
    "Annualized Return"    : rp.mean() * 12,
    "Annualized Volatility": rp.std() * np.sqrt(12),
    "Sharpe Ratio"         : (excess.mean() * 12) / (rp.std() * np.sqrt(12)),
    "Cumulative Return"    : (1 + rp).prod() - 1,
    "Min Monthly Return"   : rp.min(),
    "Max Monthly Return"   : rp.max(),
}

for k, v in metrics.items():
    print(f"{k:<25}: {v:.4%}" if k != "Sharpe Ratio" else f"{k:<25}: {v:.4f}")

Annualized Return        : 4.8590%
Annualized Volatility    : 12.6674%
Sharpe Ratio             : 0.1681
Cumulative Return        : 61.6470%
Min Monthly Return       : -25.2229%
Max Monthly Return       : 8.6483%
